In [4]:
# ============================================================
# ISYE 474/574 — Assignment 3 (Extended)
# Simulated Annealing (SA) for 1 || sum w_j T_j  (weighted tardiness)
#
# EXTENSIONS over base version:
#   • Wall-clock runtime per run  (time.perf_counter)
#   • Peak memory delta per run   (tracemalloc)
#   • Summary statistics across all 10 runs
#   • Computational-complexity (Big-O) discussion printed at the end
#
# Produces a 10-run sweep table with:
# Run, Initial Temperature, Cooling Rate, Stopping Criteria,
# Initial Schedule, Initial Solution (WT),
# Final Schedule, Final Solution (WT), % Improvement,
# Runtime (s), Peak Memory (KB)
# ============================================================

import math
import random
import time
import tracemalloc
from dataclasses import dataclass
from typing import List, Dict

try:
    import pandas as pd
except ImportError:
    pd = None


# -----------------------------
# 1) DATA (Jobs 1..8)
# -----------------------------
P = {1: 5, 2: 8, 3: 3, 4: 2, 5: 10, 6: 4, 7: 6, 8: 7}       # processing times
D = {1: 10, 2: 15, 3: 12, 4: 20, 5: 25, 6: 22, 7: 30, 8: 18}  # due dates
W = {1: 10, 2: 4, 3: 15, 4: 1, 5: 30, 6: 8, 7: 12, 8: 20}     # weights
JOBS = list(P.keys())


# -----------------------------
# 2) OBJECTIVE: total weighted tardiness
# -----------------------------
def weighted_tardiness(schedule: List[int]) -> int:
    """Compute sum_j w_j * max(C_j - d_j, 0) for a given job order."""
    t = 0
    total = 0
    for j in schedule:
        t += P[j]
        Tj = max(t - D[j], 0)
        total += W[j] * Tj
    return total


def schedule_to_str(schedule: List[int]) -> str:
    return "-".join(map(str, schedule))


# -----------------------------
# 3) CONSTRUCTION METHODS (initial solution)
# -----------------------------
def construct_initial(method: str, rng: random.Random) -> List[int]:
    """
    Construction heuristics:
      RANDOM, EDD, SPT, WSPT, MDD, WMDD
    """
    m = method.strip().upper()

    if m == "RANDOM":
        s = JOBS[:]
        rng.shuffle(s)
        return s

    if m == "EDD":
        return sorted(JOBS, key=lambda j: (D[j], j))

    if m == "SPT":
        return sorted(JOBS, key=lambda j: (P[j], j))

    if m == "WSPT":
        return sorted(JOBS, key=lambda j: (P[j] / W[j], j))

    if m == "MDD":
        remaining = set(JOBS)
        s, t = [], 0
        while remaining:
            j_star = min(remaining, key=lambda j: (max(D[j], t + P[j]), D[j], j))
            s.append(j_star)
            remaining.remove(j_star)
            t += P[j_star]
        return s

    if m == "WMDD":
        remaining = set(JOBS)
        s, t = [], 0
        while remaining:
            j_star = min(remaining, key=lambda j: (max(D[j], t + P[j]) / W[j], D[j], j))
            s.append(j_star)
            remaining.remove(j_star)
            t += P[j_star]
        return s

    raise ValueError(f"Unknown construction method: {method}")


# -----------------------------
# 4) NEIGHBOURHOOD MOVES
# -----------------------------
def neighbour(schedule: List[int], move: str, rng: random.Random) -> List[int]:
    n = len(schedule)
    s = schedule[:]
    mv = move.strip().lower()

    if n < 2:
        return s

    if mv == "swap":
        i, j = rng.sample(range(n), 2)
        s[i], s[j] = s[j], s[i]
        return s

    if mv == "adjacent":
        i = rng.randrange(n - 1)
        s[i], s[i + 1] = s[i + 1], s[i]
        return s

    if mv == "insert":
        i, j = rng.sample(range(n), 2)
        job = s.pop(i)
        s.insert(j, job)
        return s

    raise ValueError(f"Unknown move type: {move}")


# -----------------------------
# 5) STOPPING RULE
# -----------------------------
@dataclass
class StopRule:
    kind: str   # "N" or "TMIN"
    value: float


# -----------------------------
# 6) SIMULATED ANNEALING CORE (with timing + memory)
# -----------------------------
def simulated_annealing(
    T0: float,
    alpha: float,
    stop: StopRule,
    init_method: str,
    move: str,
    seed: int = 2026,
    hard_cap_iters: int = 200_000,
) -> Dict:
    """
    SA minimisation with runtime and peak-memory instrumentation.
    """
    if T0 <= 0:
        raise ValueError("T0 must be > 0.")
    if not (0 < alpha < 1):
        raise ValueError("alpha must be in (0, 1).")
    if stop.kind.upper() not in ("N", "TMIN"):
        raise ValueError("StopRule.kind must be 'N' or 'TMIN'.")

    rng = random.Random(seed)

    # ---- start memory tracking ----
    tracemalloc.start()
    snap_before = tracemalloc.take_snapshot()
    t_start = time.perf_counter()

    # Initial schedule
    s0 = construct_initial(init_method, rng)
    f0 = weighted_tardiness(s0)

    s_cur, f_cur = s0[:], f0
    s_best, f_best = s0[:], f0

    T = float(T0)
    k = 0

    while True:
        # stopping check
        if stop.kind.upper() == "N":
            if k >= int(stop.value):
                break
        else:
            if T <= float(stop.value):
                break
            if k >= hard_cap_iters:
                break

        s_new = neighbour(s_cur, move, rng)
        f_new = weighted_tardiness(s_new)
        delta = f_new - f_cur

        if delta <= 0:
            accept = True
        else:
            accept = (rng.random() < math.exp(-delta / T))

        if accept:
            s_cur, f_cur = s_new, f_new
            if f_cur < f_best:
                s_best, f_best = s_cur[:], f_cur

        T *= alpha
        k += 1

    # ---- stop timing / memory ----
    t_end = time.perf_counter()
    snap_after = tracemalloc.take_snapshot()
    tracemalloc.stop()

    runtime_s = t_end - t_start

    # Peak memory delta (KB) — compare top allocations
    stats = snap_after.compare_to(snap_before, "lineno")
    peak_mem_kb = sum(s.size_diff for s in stats if s.size_diff > 0) / 1024.0

    improvement = 0.0 if f0 == 0 else (f0 - f_best) / f0 * 100.0

    return {
        "Initial Schedule": s0,
        "Initial WT": f0,
        "Final Schedule": s_best,
        "Final WT": f_best,
        "% Improvement": improvement,
        "Iterations": k,
        "Final Temperature": T,
        "Runtime (s)": runtime_s,
        "Peak Memory (KB)": peak_mem_kb,
    }


# -----------------------------
# 7) 10-RUN SWEEP PLAN
# -----------------------------
def build_10_run_plan() -> List[Dict]:
    return [
        {"Run": 1,  "T0": 100, "alpha": 0.99,  "stop": StopRule("N", 100),  "init": "WSPT", "move": "swap"},
        {"Run": 2,  "T0": 100, "alpha": 0.99,  "stop": StopRule("N", 200),  "init": "EDD",  "move": "swap"},
        {"Run": 3,  "T0": 150, "alpha": 0.995, "stop": StopRule("N", 400),  "init": "SPT",  "move": "insert"},
        {"Run": 4,  "T0": 200, "alpha": 0.98,  "stop": StopRule("N", 600),  "init": "MDD",  "move": "insert"},
        {"Run": 5,  "T0": 75,  "alpha": 0.985, "stop": StopRule("N", 600),  "init": "WMDD", "move": "swap"},
        {"Run": 6,  "T0": 100, "alpha": 0.99,  "stop": StopRule("TMIN", 0.1), "init": "WSPT", "move": "adjacent"},
        {"Run": 7,  "T0": 150, "alpha": 0.99,  "stop": StopRule("TMIN", 0.1), "init": "EDD",  "move": "swap"},
        {"Run": 8,  "T0": 200, "alpha": 0.995, "stop": StopRule("TMIN", 0.1), "init": "SPT",  "move": "insert"},
        {"Run": 9,  "T0": 250, "alpha": 0.997, "stop": StopRule("TMIN", 0.1), "init": "MDD",  "move": "swap"},
        {"Run": 10, "T0": 80,  "alpha": 0.985, "stop": StopRule("TMIN", 0.1), "init": "WMDD", "move": "insert"},
    ]


def run_sweep(plan: List[Dict], base_seed: int = 202600) -> List[Dict]:
    results = []

    for row in plan:
        run_id = int(row["Run"])
        stop = row["stop"]

        out = simulated_annealing(
            T0=float(row["T0"]),
            alpha=float(row["alpha"]),
            stop=stop,
            init_method=row["init"],
            move=row["move"],
            seed=base_seed + run_id,
        )

        stop_str = (f"N = {int(stop.value)}"
                     if stop.kind.upper() == "N"
                     else f"Tmin = {stop.value}")

        results.append({
            "Run": run_id,
            "Initial Temperature": row["T0"],
            "Cooling Rate": row["alpha"],
            "Stopping Criteria": stop_str,
            "Init Method": row["init"],
            "Move": row["move"],

            "Initial Schedule": schedule_to_str(out["Initial Schedule"]),
            "Initial Solution": out["Initial WT"],

            "Final Schedule": schedule_to_str(out["Final Schedule"]),
            "Final Solution": out["Final WT"],

            "% Improvement": round(out["% Improvement"], 2),
            "Iterations": out["Iterations"],
            "Final Temperature": round(out["Final Temperature"], 6),

            # ---- NEW METRICS ----
            "Runtime (s)": round(out["Runtime (s)"], 6),
            "Peak Memory (KB)": round(out["Peak Memory (KB)"], 2),
        })

    return results


# -----------------------------
# 8) SUMMARY + COMPLEXITY ANALYSIS
# -----------------------------
def print_summary(results: List[Dict]):
    """Print aggregate statistics across all runs."""
    runtimes  = [r["Runtime (s)"]      for r in results]
    memories  = [r["Peak Memory (KB)"] for r in results]
    finals    = [r["Final Solution"]   for r in results]
    improves  = [r["% Improvement"]    for r in results]
    iters     = [r["Iterations"]       for r in results]

    print("\n" + "=" * 70)
    print("SUMMARY STATISTICS ACROSS 10 RUNS")
    print("=" * 70)
    print(f"  {'Metric':<25} {'Min':>10} {'Mean':>10} {'Max':>10}")
    print(f"  {'-'*25} {'-'*10} {'-'*10} {'-'*10}")

    for label, vals in [
        ("Final WT",          finals),
        ("% Improvement",     improves),
        ("Iterations",        iters),
        ("Runtime (s)",       runtimes),
        ("Peak Memory (KB)",  memories),
    ]:
        mn  = min(vals)
        mx  = max(vals)
        avg = sum(vals) / len(vals)
        print(f"  {label:<25} {mn:>10.4f} {avg:>10.4f} {mx:>10.4f}")

    # Identify best run by final WT
    best = min(results, key=lambda r: r["Final Solution"])
    print(f"\n  Best run: Run {best['Run']}  "
          f"(WT = {best['Final Solution']}, "
          f"{best['% Improvement']:.2f}% improvement, "
          f"{best['Runtime (s)']:.4f}s, "
          f"{best['Peak Memory (KB)']:.2f} KB)")

    # Identify most efficient (best WT per second)
    # Lower ratio = better quality per unit time
    for r in results:
        r["_efficiency"] = r["Final Solution"] * r["Runtime (s)"]
    efficient = min(results, key=lambda r: r["_efficiency"])
    print(f"  Most efficient: Run {efficient['Run']}  "
          f"(WT×time product = {efficient['_efficiency']:.4f})")
    # clean up temp key
    for r in results:
        del r["_efficiency"]


def print_complexity_analysis():
    """
    Print Big-O discussion for the SA algorithm.
    """
    print("\n" + "=" * 70)
    print("COMPUTATIONAL COMPLEXITY ANALYSIS (Big-O)")
    print("=" * 70)
    analysis = """
  Let:
    n = number of jobs
    K = number of SA iterations (either fixed N, or derived from T0, alpha, Tmin)

  1) Construction Heuristic (one-time cost)
     ─────────────────────────────────────
     • RANDOM, EDD, SPT, WSPT : O(n log n)   [sorting-based]
     • MDD, WMDD              : O(n^2)        [greedy selection at each of n steps]

  2) Objective Evaluation — weighted_tardiness()
     ─────────────────────────────────────
     • O(n) per call — single pass through the schedule.

  3) Neighbourhood Move — neighbour()
     ─────────────────────────────────────
     • swap / adjacent : O(n)   [copy list + constant-time swap]
     • insert          : O(n)   [copy + pop + insert on a Python list]

  4) SA Main Loop
     ─────────────────────────────────────
     Each iteration performs:
       – One neighbour generation : O(n)
       – One objective evaluation : O(n)
       – Acceptance decision      : O(1)
       – Cooling step             : O(1)
     → Per iteration: O(n)
     → Total loop:    O(K · n)

  5) Iteration Count K
     ─────────────────────────────────────
     • Fixed N stopping    : K = N  (directly controlled)
     • Tmin stopping       : K = ceil( log(Tmin/T0) / log(alpha) )
       Example: T0=100, alpha=0.99, Tmin=0.1 → K = ceil(log(0.001)/log(0.99)) ≈ 688
       Example: T0=250, alpha=0.997, Tmin=0.1 → K = ceil(log(0.0004)/log(0.997)) ≈ 2604

  6) Overall Complexity
     ─────────────────────────────────────
     O(n^2)  +  O(K · n)
     [init]     [SA loop]

     For small n (here n=8), the loop dominates and complexity is
     effectively O(K).  For large n the objective evaluation inside
     the loop would dominate, giving O(K · n).

  7) Space Complexity
     ─────────────────────────────────────
     O(n) — only a constant number of schedule copies (current,
     best, new) each of length n, plus the job data arrays.

  8) Practical Scaling Note
     ─────────────────────────────────────
     With n=8 jobs, all runs complete in microseconds to low
     milliseconds.  The algorithm scales linearly in K for a
     fixed n.  If n grows large, using an incremental (delta)
     objective evaluation after a swap can reduce the per-
     iteration cost from O(n) to O(1), improving the loop
     to O(K) regardless of n.
"""
    print(analysis)


# -----------------------------
# 9) MAIN
# -----------------------------
if __name__ == "__main__":
    plan = build_10_run_plan()
    res = run_sweep(plan)

    if pd is not None:
        df = pd.DataFrame(res)

        cols = [
            "Run", "Initial Temperature", "Cooling Rate", "Stopping Criteria",
            "Init Method", "Move",
            "Initial Schedule", "Initial Solution",
            "Final Schedule", "Final Solution", "% Improvement",
            "Iterations", "Final Temperature",
            "Runtime (s)", "Peak Memory (KB)",
        ]
        df = df[cols]

        pd.set_option("display.max_colwidth", None)
        pd.set_option("display.width", 220)
        pd.set_option("display.max_columns", None)

        print(df.to_string(index=False))

        df.to_csv("assignment3_SA_extended_results.csv", index=False)
        print("\nSaved: assignment3_SA_extended_results.csv")
    else:
        for r in res:
            print(r)

    # Print summary and complexity analysis
    print_summary(res)
    print_complexity_analysis()

 Run  Initial Temperature  Cooling Rate Stopping Criteria Init Method     Move Initial Schedule  Initial Solution  Final Schedule  Final Solution  % Improvement  Iterations  Final Temperature  Runtime (s)  Peak Memory (KB)
   1                  100         0.990           N = 100        WSPT     swap  3-5-8-1-6-7-2-4               443 3-1-8-5-6-7-4-2             253          42.89         100          36.603234     0.008779              4.45
   2                  100         0.990           N = 200         EDD     swap  1-3-2-8-4-6-5-7               765 3-1-8-5-6-7-2-4             253          66.93         200          13.397967     0.017575              1.39
   3                  150         0.995           N = 400         SPT   insert  4-3-6-1-7-8-2-5               900 1-3-8-5-7-6-2-4             253          71.89         400          20.198706     0.035935              1.18
   4                  200         0.980           N = 600         MDD   insert  1-3-2-4-6-8-7-5             